# Training

In [1]:
!pip install -q ml-collections


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import jax

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
import tensorflow as tf
# Set tensorflow to CPU
tf.config.set_visible_devices([], 'TPU')
tf.config.set_visible_devices([], 'GPU')

2026-05-12 18:50:03.377591: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
import os
import urllib.request
from urllib.error import HTTPError
import ml_collections
import numpy as np
import jax
from jax import numpy as jnp
from jax.sharding import Mesh
from tokenizers import ByteLevelBPETokenizer

main_rng_key = jax.random.key(18)

E0000 00:00:1778611806.375972      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


In [5]:
!rm -rf de_tokenizer_20_000_vocab_size_model
!rm -rf en_tokenizer_20_000_vocab_size_model
!rm -rf log_dir
!rm -f configs.py tokenizer.py data.py fsdp_model.py fsdp_model_utils.py fsdp_training_utils.py


!mkdir de_tokenizer_20_000_vocab_size_model
!mkdir en_tokenizer_20_000_vocab_size_model

/usr/local/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [6]:
base_url = "https://raw.githubusercontent.com/MiguelSteph/transformer-from-scratch/fsdp/"
def download_file_from_github(file_path: str, file_name: str):
    if not os.path.isfile(file_name):
        file_url = base_url + file_path
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_name)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file directly from the GitHub repository:\n", e)

file_paths = [
    'configs/configs.py',
    'data/tokenizer.py',
    'data/data.py',
    'models/fsdp_model.py',
    'models/fsdp_model_utils.py',
    'training/fsdp_training_utils.py',
    'data/en_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/en_tokenizer_20_000_vocab_size_model/vocab.json',
    'data/de_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/de_tokenizer_20_000_vocab_size_model/vocab.json',
]
file_names = [
    'configs.py',
    'tokenizer.py',
    'data.py',
    'fsdp_model.py',
    'fsdp_model_utils.py',
    'fsdp_training_utils.py',
    'en_tokenizer_20_000_vocab_size_model/merges.txt',
    'en_tokenizer_20_000_vocab_size_model/vocab.json',
    'de_tokenizer_20_000_vocab_size_model/merges.txt',
    'de_tokenizer_20_000_vocab_size_model/vocab.json',
]

for file_path, file_name in zip(file_paths, file_names): 
    download_file_from_github(file_path, file_name)

In [7]:
from configs import get_configs
from fsdp_model import create_transformer_module
from fsdp_training_utils import train_and_evaluate, get_dataset_iterator, fsdp_init, generate_random_batch
from data import load_preprocessed_dataset

base_configs = get_configs()
config = ml_collections.ConfigDict(base_configs)
config.data.test_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/test.tfrecord'
config.data.validation_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/validation.tfrecord'
config.data.train_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/train.tfrecord'

train_ds = load_preprocessed_dataset(config.data.train_ds_path, config.data.max_seq_len)
validation_ds = load_preprocessed_dataset(config.data.validation_ds_path, config.data.max_seq_len)
test_ds = load_preprocessed_dataset(config.data.test_ds_path, config.data.max_seq_len)

de_tokenizer = ByteLevelBPETokenizer(config.data.de_tokenizer_model_path + '/vocab.json',
                                     config.data.de_tokenizer_model_path + '/merges.txt')
de_tokenizer.add_special_tokens(list(config.data.special_tokens))
en_tokenizer = ByteLevelBPETokenizer(config.data.en_tokenizer_model_path + '/vocab.json',
                                     config.data.en_tokenizer_model_path + '/merges.txt')
en_tokenizer.add_special_tokens(list(config.data.special_tokens))

enc_pad_id = de_tokenizer.encode('<|pad|>').ids[0]
dec_pad_id = en_tokenizer.encode('<|pad|>').ids[0]

ERROR:absl:Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/site-pa

In [8]:
!rm -rf log_dir 
!rm -f log_dir.zip

/usr/local/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [9]:
config

data:
  batch_size: 64
  de_tokenizer_model_path: de_tokenizer_20_000_vocab_size_model
  en_tokenizer_model_path: en_tokenizer_20_000_vocab_size_model
  max_seq_len: 100
  special_tokens:
  - <|pad|>
  - <|startoftext|>
  - <|endoftext|>
  test_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/test.tfrecord
  train_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/train.tfrecord
  validation_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/validation.tfrecord
  vocab_size: 20000
fsdp:
  data_axis: data
  min_weight_size: 256
model:
  d_proj: 64
  dropout: 0.1
  emb_dim: 512
  ff_d_inner_factor: 4
  num_blocks: 6
  num_heads: 8
optimizer:
  base_lr: 0.0001
  steps_per_epochs: 30000
  training_epochs: 24
  warmup_epochs: 4
training_output:
  checkpoint_path: log_dir/checkpoints
  metric_path: log_dir/metrics
  trace_path: log_dir/traces

## Training

In [10]:
# Create the mesh
device_array = np.array(jax.devices())
mesh = Mesh(device_array, (config.fsdp.data_axis,))

# Create the model
model = create_transformer_module(config, enc_pad_id, dec_pad_id)

# Create the train state and get the sharding specs
state, state_specs = fsdp_init(
    model,
    mesh,
    config,
    dec_pad_id,
)

# Call Train_and_evaluate
state = train_and_evaluate(model, 
                           mesh,
                           state,
                           state_specs,
                           config,
                           main_rng_key,
                           train_ds,
                           validation_ds,
                           log_dir_prefix=None,
                           dec_pad_id=dec_pad_id)

Epoch 1


  0%|          | 0/30000 [00:00<?, ?it/s]

Training:    Loss: 7.0247483253479    Accuracy: 0.14058493077754974
Validation:  Loss: 5.97496223449707    Accuracy: 0.18499086797237396
Epoch 2


  0%|          | 0/30000 [00:00<?, ?it/s]

Training:    Loss: 5.691450119018555    Accuracy: 0.25337332487106323
Validation:  Loss: 4.6513752937316895    Accuracy: 0.3011910915374756
Epoch 3


...

Training:    Loss: 3.116020917892456    Accuracy: 0.607310950756073
Validation:  Loss: 2.0056862831115723    Accuracy: 0.6130990386009216
Epoch 23


  0%|          | 0/30000 [00:00<?, ?it/s]

Training:    Loss: 3.105471611022949    Accuracy: 0.6090610027313232
Validation:  Loss: 1.9964261054992676    Accuracy: 0.6134024858474731
Epoch 24


  0%|          | 0/30000 [00:00<?, ?it/s]

Training:    Loss: 3.0998857021331787    Accuracy: 0.6098456382751465
Validation:  Loss: 1.992640733718872    Accuracy: 0.6142928600311279
